# Entendimento do negócio


In [1]:
# Importações
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import ee
import requests
import re

import urllib3
import zipfile
import shutil
import json
import gc

import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')  # backend não-interativo, evita crash de GUI/DLL no Windows
import matplotlib.pyplot as plt
import numpy as np

from glob import glob
from io import BytesIO
from shapely.geometry import box

print("OK")

OK


In [2]:
# Roda o comando no terminal : earthengine authenticate --auth_mode=notebook
ee.Authenticate()
ee.Initialize(project="spatial-yew-490017-r3")
print(ee.String("Hello from the Earth Engine servers!").getInfo())

Hello from the Earth Engine servers!


In [3]:
# mostrar todas as colunas
pd.set_option("display.max_columns", None)

# não quebrar a largura da tabela
pd.set_option("display.expand_frame_repr", False)

# opcional: aumentar a largura máxima exibida
pd.set_option("display.width", 1000)

In [4]:
os.environ["GDAL_DATA"] = os.path.join(
    os.environ["CONDA_PREFIX"], "Library", "share", "gdal"
)
os.environ["PROJ_LIB"] = os.path.join(
    os.environ["CONDA_PREFIX"], "Library", "share", "proj"
)

In [5]:
FILES_DIR = "files"
os.makedirs(FILES_DIR, exist_ok=True)

# Entendimento dos dados


In [ ]:
# Download UCs boundaries via WFS (INDE/ICMBio)

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

ucs_dir = os.path.join(FILES_DIR, "limites_ucs")
os.makedirs(ucs_dir, exist_ok=True)

uc_typename = "ICMBio:limiteucsfederais_a"
uc_output_path = os.path.join(ucs_dir, "limites_ucs.geojson")

if os.path.exists(uc_output_path):
    print(f"Already downloaded: {uc_output_path}")
else:
    wfs_url = (
        "https://geoservicos.inde.gov.br/geoserver/ICMBio/ows"
        f"?service=WFS&version=2.0.0&request=GetFeature"
        f"&typeName={uc_typename}&outputFormat=application/json"
    )
    response = requests.get(wfs_url, timeout=180, verify=False)
    response.raise_for_status()
    with open(uc_output_path, "wb") as f:
        f.write(response.content)
    print(f"Saved: {uc_output_path}")

gdf_ucs = gpd.read_file(uc_output_path)

print("Shape:", gdf_ucs.shape)
print("Columns:", gdf_ucs.columns.tolist())
print("CRS:", gdf_ucs.crs)

In [ ]:
# Reprojetar UCs para CRS métrico

METRIC_CRS = "EPSG:5880"  # SIRGAS 2000 / Brazil Polyconic
CELL_SIZE_M = 5000  # 5km x 5km, ajustável

gdf_ucs_metric = gdf_ucs.to_crs(METRIC_CRS)
gdf_ucs_metric["geometry"] = gdf_ucs_metric.geometry.buffer(0)

print("Geometrias inválidas em gdf_ucs_metric:", (~gdf_ucs_metric.is_valid).sum())

In [ ]:
# INPE hotspots

INPE_YEARS = [2003, 2025]

inpe_years_to_download = (
    INPE_YEARS
    if len(INPE_YEARS) == 1
    else list(range(min(INPE_YEARS), max(INPE_YEARS) + 1))
)
print("INPE years queued:", inpe_years_to_download)

In [ ]:
# INPE FOCO DE CALOR - downlaod de range de ano
inpe_dir = os.path.join(FILES_DIR, "inpe_focos")
os.makedirs(inpe_dir, exist_ok=True)

inpe_downloaded_files = {}

for year in inpe_years_to_download:

    csv_filename = f"focos_br_ref_{year}.csv"
    csv_path = os.path.join(inpe_dir, csv_filename)

    if os.path.exists(csv_path):
        inpe_downloaded_files[year] = csv_path
        continue

    zip_url = f"https://dataserver-coids.inpe.br/queimadas/queimadas/focos/csv/anual/Brasil_todos_sats/focos_br_todos-sats_{year}.zip"

    try:
        response = requests.get(zip_url, timeout=180)
        response.raise_for_status()

        tmp_dir = os.path.join(inpe_dir, f"tmp_{year}")
        os.makedirs(tmp_dir, exist_ok=True)

        with zipfile.ZipFile(BytesIO(response.content)) as z:
            z.extractall(tmp_dir)

        found_csvs = glob(os.path.join(tmp_dir, "**", "*.csv"), recursive=True)

        if found_csvs:
            shutil.move(found_csvs[0], csv_path)
            inpe_downloaded_files[year] = csv_path
        else:
            print(f"[{year}] No CSV found in zip")

        shutil.rmtree(tmp_dir, ignore_errors=True)

    except requests.exceptions.RequestException:
        print(f"[{year}] Download failed")

print("Done:", list(inpe_downloaded_files.keys()))

In [ ]:
# filtro contra a geometria das UCs
colunas_necessarias = [
    "latitude",
    "longitude",
    "data_pas",
    "satelite",
    "bioma",
    "estado",
    "municipio",
    "frp",
    "risco_fogo",
    "numero_dias_sem_chuva",
]

dtypes_reduzidos = {
    "satelite": "category",
    "bioma": "category",
    "estado": "category",
    "municipio": "category",
    "frp": "float32",
    "risco_fogo": "float32",
    "numero_dias_sem_chuva": "float32",
    "latitude": "float64",
    "longitude": "float64",
}

inpe_df_list = []

for year, path in inpe_downloaded_files.items():

    df_year = pd.read_csv(path, usecols=colunas_necessarias, dtype=dtypes_reduzidos)

    # geometria temporária, só para o teste espacial deste ano — descartada logo em seguida
    points_year = gpd.GeoDataFrame(
        df_year[["latitude", "longitude"]],
        geometry=gpd.points_from_xy(df_year["longitude"], df_year["latitude"]),
        crs="EPSG:4326",
    ).to_crs(METRIC_CRS)

    matched = gpd.sjoin(
        points_year,
        gdf_ucs_metric[["cnuc", "geometry"]],
        how="inner",
        predicate="within",
    )

    df_year_filtered = df_year.loc[matched.index].copy()
    df_year_filtered["file_year"] = year

    print(
        f"[{year}] Total: {len(df_year)} | Dentro de alguma UC: {len(df_year_filtered)}"
    )

    inpe_df_list.append(df_year_filtered)

    del df_year, points_year, matched, df_year_filtered
    gc.collect()

df_inpe = pd.concat(inpe_df_list, ignore_index=True)
del inpe_df_list
gc.collect()

print("\nShape final (já filtrado por UC real):", df_inpe.shape)
print("Columns:", df_inpe.columns.tolist())
print("Years present:", sorted(df_inpe["file_year"].unique()))

In [ ]:
# AAF — download via ArcGIS FeatureServer (anos 2010-2026)

BASE_URL = "https://services3.arcgis.com/KYEMegXJrTiWSYWk/arcgis/rest/services"
PAGE_SIZE = 2000

aaf_urls_by_year = {}

# Padrão dinâmico — anos 2010 a 2022 seguem a mesma convenção de nome
for year in range(2010, 2023):
    aaf_urls_by_year[year] = (
        f"{BASE_URL}/db_geo_compartilhado_dmif_fogo_aaf_{year}/FeatureServer/0"
    )

# Exceções — nomes de serviço mudaram ano a ano a partir de 2023
aaf_urls_by_year[2023] = f"{BASE_URL}/AAF_2023_DGEO_ICMBIO_oficial/FeatureServer/0"
aaf_urls_by_year[2024] = f"{BASE_URL}/AAF_2024_DGEO_ICMBIO_oficial/FeatureServer/0"
aaf_urls_by_year[2025] = f"{BASE_URL}/AAF_2025_DGEO_oficial/FeatureServer/0"
aaf_urls_by_year[2026] = f"{BASE_URL}/AAF_2026/FeatureServer/0"

print("Total de anos mapeados:", len(aaf_urls_by_year))

In [ ]:
# Download com paginação, salvando um GeoJSON por ano

aaf_dir = os.path.join(FILES_DIR, "aaf")
os.makedirs(aaf_dir, exist_ok=True)

aaf_downloaded_files = {}

for year, base_url in aaf_urls_by_year.items():

    output_path = os.path.join(aaf_dir, f"aaf_{year}.geojson")

    if os.path.exists(output_path):
        aaf_downloaded_files[year] = output_path
        continue

    # Paginação — segue baixando enquanto o servidor retornar página cheia
    all_features = []
    offset = 0

    while True:
        query_url = (
            f"{base_url}/query?where=1%3D1&outFields=*&outSR=4326&f=geojson"
            f"&resultOffset={offset}&resultRecordCount={PAGE_SIZE}"
        )
        response = requests.get(query_url, timeout=120)
        response.raise_for_status()
        page_data = response.json()

        features = page_data.get("features", [])
        if not features:
            break

        all_features.extend(features)

        if len(features) < PAGE_SIZE:
            break

        offset += PAGE_SIZE

    # Salvar como GeoJSON válido
    final_geojson = {"type": "FeatureCollection", "features": all_features}

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_geojson, f)

    aaf_downloaded_files[year] = output_path
    print(f"[{year}] Salvo — {len(all_features)} registros")

print("\nAnos baixados:", list(aaf_downloaded_files.keys()))

In [ ]:
# Carregar todos os anos, checando consistência de colunas antes de concatenar

aaf_gdf_list = []
reference_columns = None

for year, path in aaf_downloaded_files.items():
    gdf_year = gpd.read_file(path)

    if reference_columns is None:
        reference_columns = set(gdf_year.columns)
    else:
        diff = set(gdf_year.columns).symmetric_difference(reference_columns)
        if diff:
            print(f"[{year}] Diferença de colunas em relação ao primeiro ano: {diff}")

    gdf_year["file_year"] = year
    aaf_gdf_list.append(gdf_year)

gdf_aaf = gpd.GeoDataFrame(pd.concat(aaf_gdf_list, ignore_index=True))


print("Shape:", gdf_aaf.shape)
print("Colunas:", gdf_aaf.columns.tolist())
print("Anos presentes:", sorted(gdf_aaf["file_year"].unique()))

In [ ]:
# Harmonizar colunas divergentes entre eras do schema

gdf_aaf["event_year"] = gdf_aaf["file_year"]

# converter cada fonte para datetime
date_from_old_schema = pd.to_datetime(gdf_aaf["data"], unit="ms", errors="coerce")
date_from_new_schema = pd.to_datetime(gdf_aaf["data_img"], errors="coerce")
gdf_aaf["event_date"] = date_from_old_schema.combine_first(date_from_new_schema)

#  recupera datas quando 'data' e 'data_img' estão ausentes
julian_fallback = pd.to_datetime(
    gdf_aaf["file_year"].astype(str), format="%Y", errors="coerce"
) + pd.to_timedelta(gdf_aaf["juliano"] - 1, unit="D")
gdf_aaf["event_date"] = gdf_aaf["event_date"].combine_first(julian_fallback)
gdf_aaf["event_month"] = gdf_aaf["event_date"].dt.month

# coalesce entre nomenclatura abreviada e completa
gdf_aaf["shape_area"] = gdf_aaf["Shape__Are"].combine_first(gdf_aaf["Shape__Area"])
gdf_aaf["shape_length"] = gdf_aaf["Shape__Len"].combine_first(gdf_aaf["Shape__Length"])

# Descartar eventos sem data válida
n_before = len(gdf_aaf)
gdf_aaf = gdf_aaf[gdf_aaf["event_date"].notna()].copy()

print(
    f"Eventos descartados por falta de data: {n_before - len(gdf_aaf)} ({(n_before - len(gdf_aaf)) / n_before * 100:.1f}%)"
)
print(f"Total de eventos final: {len(gdf_aaf)}")
print(f"Nulos em shape_area: {gdf_aaf['shape_area'].isna().sum()}")

# Preparação dos dados


In [ ]:
# Reprojetar AAF

gdf_aaf_metric = gdf_aaf.to_crs(METRIC_CRS)
print("CRS AAF após reprojeção:", gdf_aaf_metric.crs)

In [ ]:
gdf_aaf_metric["geometry"] = gdf_aaf_metric.geometry.buffer(0)
print("Geometrias inválidas em gdf_aaf_metric:", (~gdf_aaf_metric.is_valid).sum())

In [ ]:
# Descartar volume irrelevante
n_before = len(gdf_aaf_metric)
gdf_aaf_metric = gdf_aaf_metric[gdf_aaf_metric.is_valid].copy()

print(
    f"\nRegistros descartados por geometria inválida: {n_before - len(gdf_aaf_metric)}"
)
print(f"Total de eventos final: {len(gdf_aaf_metric)}")

In [ ]:
# Remover as poucas duplicatas geométricas reais
n_before = len(gdf_aaf_metric)
gdf_aaf_metric = gdf_aaf_metric[
    ~gdf_aaf_metric.geometry.apply(lambda g: g.wkb).duplicated()
].copy()

print(f"Duplicatas geométricas removidas: {n_before - len(gdf_aaf_metric)}")
print(f"Total final: {len(gdf_aaf_metric)}")

In [ ]:
# Origem global fixa

global_minx, global_miny, _, _ = gdf_ucs_metric.total_bounds

grid_records = []

for idx, row in gdf_ucs_metric.iterrows():
    uc_geom = row.geometry
    minx, miny, maxx, maxy = uc_geom.bounds

    i_start = int((minx - global_minx) // CELL_SIZE_M)
    i_end = int((maxx - global_minx) // CELL_SIZE_M) + 1
    j_start = int((miny - global_miny) // CELL_SIZE_M)
    j_end = int((maxy - global_miny) // CELL_SIZE_M) + 1

    for i in range(i_start, i_end):
        for j in range(j_start, j_end):
            cell_x = global_minx + i * CELL_SIZE_M
            cell_y = global_miny + j * CELL_SIZE_M
            cell = box(cell_x, cell_y, cell_x + CELL_SIZE_M, cell_y + CELL_SIZE_M)

            if uc_geom.intersects(cell):
                clipped = uc_geom.intersection(cell)
                if not clipped.is_empty:
                    grid_records.append({"cell_id": f"{i}_{j}", "geometry": clipped})

gdf_grid_raw = gpd.GeoDataFrame(grid_records, crs=METRIC_CRS)
print("Peças de célula geradas:", len(gdf_grid_raw))

gdf_grid = gdf_grid_raw.dissolve(by="cell_id", as_index=False)
del gdf_grid_raw, grid_records
gc.collect()

print("Total de células globais únicas:", len(gdf_grid))

In [ ]:
# Tabela de associação de célula UC.

cell_uc_membership = gpd.sjoin(
    gdf_grid, gdf_ucs_metric[["cnuc", "geometry"]], how="left", predicate="intersects"
)[["cell_id", "cnuc"]]

cells_per_uc_count = cell_uc_membership.groupby("cell_id").size()
print("Células pertencentes a mais de um UC:", (cells_per_uc_count > 1).sum())

In [ ]:
# Atribua eventos AAF a células da grade via sobreposição

gdf_aaf_metric = gpd.overlay(
    gdf_aaf_metric, gdf_grid[["cell_id", "geometry"]], how="intersection"
)

gdf_aaf_metric["area_ha_cell"] = gdf_aaf_metric.geometry.area / 10000

print("Total de linhas após a sobreposição:", len(gdf_aaf_metric))

In [ ]:
n_before = len(df_inpe)
df_inpe = df_inpe.drop_duplicates(
    subset=["latitude", "longitude", "data_pas", "satelite", "frp"]
).copy()
print(f"Duplicatas removidas de df_inpe: {n_before - len(df_inpe)}")
print(f"Total final: {len(df_inpe)}")

In [ ]:
# Filtro em cascata para o INPE

gdf_inpe_points = gpd.GeoDataFrame(
    df_inpe,
    geometry=gpd.points_from_xy(df_inpe["longitude"], df_inpe["latitude"]),
    crs="EPSG:4326",
).to_crs(METRIC_CRS)

gdf_inpe_with_cell = gpd.sjoin(
    gdf_inpe_points, gdf_grid[["cell_id", "geometry"]], how="inner", predicate="within"
)

print("Total de pontos:", len(gdf_inpe_points))
print("Total dentro de alguma célula:", len(gdf_inpe_with_cell))

del gdf_inpe_points
gc.collect()

In [ ]:
hotspots_uc = gdf_inpe_with_cell.merge(cell_uc_membership, on="cell_id", how="left")

print("Focos com UC associada:", hotspots_uc["cnuc"].notna().sum())
print("Focos sem UC associada:", hotspots_uc["cnuc"].isna().sum())

In [ ]:
# Corrigir valor sentinela -999 (INPE) 

gdf_inpe_with_cell['risco_fogo'] = gdf_inpe_with_cell['risco_fogo'].replace(-999, np.nan)
gdf_inpe_with_cell['numero_dias_sem_chuva'] = gdf_inpe_with_cell['numero_dias_sem_chuva'].replace(-999, np.nan)

# Corrigir string "None" tratada como categoria válida em 'classe' (AAF)
gdf_aaf_metric['classe'] = gdf_aaf_metric['classe'].replace('None', np.nan)

In [ ]:
# Harmonizar classe (2020-2024) e acao (2025-2026) em uma única coluna de tipo de evento

harmonization_map = {
    'aceiro': 'aceiro',
    'fogo natural': 'natural',
    'raio': 'natural',
    'gestao de ignicao natural': 'natural',
    'incendio': 'incendio',
    'indigena': 'antropica',
    'gestao de ignicao antropica': 'antropica',
    'queima por indigenas isolados': 'antropica',
    'outros': 'outros',
    'queima controlada': 'queima controlada',
    'queima prescrita': 'queima prescrita',
}

gdf_aaf_metric['event_type'] = (
    gdf_aaf_metric['classe']
    .combine_first(gdf_aaf_metric['acao'])
    .map(harmonization_map)
)

In [ ]:
# Descartar coluna residual do sjoin, sem uso analítico

gdf_inpe_with_cell = gdf_inpe_with_cell.drop(columns=['index_right'], errors='ignore')

# Salvar eventos únicos do AAF como dataset separado
# (evita contar o mesmo evento várias vezes por causa do recorte por célula)

event_key_cols = ['cnuc', 'area_ha', 'event_date', 'satelite']
aaf_events_unique = gdf_aaf_metric.drop_duplicates(subset=event_key_cols).copy()

print("Linhas recortadas (por célula):", len(gdf_aaf_metric))
print("Eventos únicos:", len(aaf_events_unique))

In [ ]:
gdf_grid.head(5)

In [ ]:
cell_uc_membership.head(5)

In [ ]:
gdf_aaf_metric.head(3)

In [ ]:
aaf_events_unique.head(3)

In [ ]:
gdf_inpe_with_cell.head(5)

In [ ]:
gdf_ucs_metric.head(3)

In [ ]:
# Nulos por coluna — gdf_aaf_metric

pd.DataFrame(
    {
        "n_nulos": gdf_aaf_metric.isna().sum(),
        "pct_nulos": (gdf_aaf_metric.isna().sum() / len(gdf_aaf_metric) * 100).round(2),
    }
).sort_values("pct_nulos", ascending=False)

In [ ]:
# Nulos por coluna — gdf_inpe_with_cell

pd.DataFrame(
    {
        "n_nulos": gdf_inpe_with_cell.isna().sum(),
        "pct_nulos": (gdf_inpe_with_cell.isna().sum() / len(gdf_inpe_with_cell) * 100).round(2),
    }
).sort_values("pct_nulos", ascending=False)

In [ ]:
# Persistir os datasets finais, já tratados, em Parquet

OUTPUTS_DIR = os.path.join(FILES_DIR, "prepared")
os.makedirs(OUTPUTS_DIR, exist_ok=True)

gdf_grid.to_parquet(os.path.join(OUTPUTS_DIR, "grid.parquet"))
cell_uc_membership.to_parquet(os.path.join(OUTPUTS_DIR, "cell_uc_membership.parquet"))
gdf_aaf_metric.to_parquet(os.path.join(OUTPUTS_DIR, "aaf_clipped.parquet"))
aaf_events_unique.to_parquet(os.path.join(OUTPUTS_DIR, "aaf_events_unique.parquet"))
gdf_inpe_with_cell.to_parquet(os.path.join(OUTPUTS_DIR, "inpe_with_cell.parquet"))
gdf_ucs_metric.to_parquet(os.path.join(OUTPUTS_DIR, "ucs_metric.parquet"))

print("Arquivos salvos em:", OUTPUTS_DIR)
for f in os.listdir(OUTPUTS_DIR):
    path = os.path.join(OUTPUTS_DIR, f)
    print(f"  {f}: {os.path.getsize(path) / 1e6:.1f} MB")

# Análise exploratória de dados (EDA)


In [6]:
OUTPUTS_DIR = os.path.join(FILES_DIR, "prepared")

gdf_grid = gpd.read_parquet(os.path.join(OUTPUTS_DIR, "grid.parquet"))
cell_uc_membership = pd.read_parquet(os.path.join(OUTPUTS_DIR, "cell_uc_membership.parquet"))
gdf_aaf_metric = gpd.read_parquet(os.path.join(OUTPUTS_DIR, "aaf_clipped.parquet"))
aaf_events_unique = gpd.read_parquet(os.path.join(OUTPUTS_DIR, "aaf_events_unique.parquet"))
gdf_inpe_with_cell = gpd.read_parquet(os.path.join(OUTPUTS_DIR, "inpe_with_cell.parquet"))
gdf_ucs_metric = gpd.read_parquet(os.path.join(OUTPUTS_DIR, "ucs_metric.parquet"))

print("Grid:", gdf_grid.shape)
print("Cell-UC membership:", cell_uc_membership.shape)
print("AAF (recortado por célula):", gdf_aaf_metric.shape)
print("AAF (eventos únicos):", aaf_events_unique.shape)
print("INPE:", gdf_inpe_with_cell.shape)
print("UCs:", gdf_ucs_metric.shape)

Grid: (82045, 2)
Cell-UC membership: (85195, 2)
AAF (recortado por célula): (83807, 41)
AAF (eventos únicos): (32026, 41)
INPE: (2565532, 13)
UCs: (347, 22)


In [20]:
gdf_aaf_metric.head(3)

,pk_aaf,cnuc,nome_uc,area_ha,data_img,classe,satelite,obs,juliano,Shape__Area,Shape__Length,file_year,FID,data,ano,mes_nome,mes_num,categoria,local,area_uc,area_ent,bioma,gr_nome,ngi,cr,GlobalID,ct,acao,tipo,Shape__Are,Shape__Len,GlobalID_2,event_year,event_date,event_month,shape_area,shape_length,cell_id,geometry,area_ha_cell,event_type
0,1.0,0000.00.0027,APA Morro da Pedreira,217.3214,2011-08-17,NaN,modis_aqua,NaN,229.0,0.000187,0.117502,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,2011,2011-08-17,8.0,0.000187,0.117502,658_323,"POLYGON ((6089436.189 7853815.485, 6089211.11 ...",11.464956,NaN
1,1.0,0000.00.0027,APA Morro da Pedreira,217.3214,2011-08-17,NaN,modis_aqua,NaN,229.0,0.000187,0.117502,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,2011,2011-08-17,8.0,0.000187,0.117502,658_324,"POLYGON ((6089429.36 7854887.082, 6089457.32 7...",3.045426,NaN
2,1.0,0000.00.0027,APA Morro da Pedreira,217.3214,2011-08-17,NaN,modis_aqua,NaN,229.0,0.000187,0.117502,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,2011,2011-08-17,8.0,0.000187,0.117502,659_323,"POLYGON ((6089730.076 7854445.769, 6090059.887...",183.377438,NaN


In [7]:
print("Linhas em gdf_ucs_metric:", len(gdf_ucs_metric))
print("cnuc únicos:", gdf_ucs_metric['cnuc'].nunique())

Linhas em gdf_ucs_metric: 347
cnuc únicos: 347


In [8]:
months_label = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

def format_thousands(values):
    # formata rótulos com separador de milhar no padrão brasileiro
    return [f'{v:,.0f}'.replace(',', '.') for v in values]

### Ranking por UC — área, eventos, bioma, tamanho

In [ ]:
# Top 10 UCs — área queimada real x total de focos de calor

uc_names = gdf_ucs_metric[['cnuc', 'nomeuc']].drop_duplicates()

aaf_area_by_uc = gdf_aaf_metric.groupby('cnuc')['area_ha_cell'].sum().rename('aaf_area_ha')

# Selecionar só 'cell_id' antes do merge — evita copiar geometria/colunas extras desnecessárias
hotspot_count_by_uc = (
    gdf_inpe_with_cell[['cell_id']]
    .merge(cell_uc_membership, on='cell_id', how='left')
    .groupby('cnuc').size().rename('hotspot_count')
)

top10_area = (
    pd.concat([aaf_area_by_uc, hotspot_count_by_uc], axis=1)
    .fillna(0)
    .merge(gdf_ucs_metric[['cnuc', 'nomeuc']].drop_duplicates(), on='cnuc', how='left')  # merge leve, sem geometria
    .nlargest(10, 'aaf_area_ha')
)

fig, ax1 = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#FFFDF5')
ax1.set_facecolor('#FFFDF5')

x = np.arange(len(top10_area))
width = 0.4

bars1 = ax1.bar(x - width/2, top10_area['aaf_area_ha'], width, color='crimson', label='Área queimada AAF (ha)')
ax1.bar_label(bars1, labels=format_thousands(top10_area['aaf_area_ha']), fontsize=9, rotation=45, padding=3)
ax1.set_ylabel('Área queimada AAF (ha)', color='crimson')
ax1.set_xticks(x)
ax1.set_xticklabels(top10_area['nomeuc'], rotation=45, ha='right')
ax1.grid(axis='y', linestyle='--', alpha=0.4)

ax2 = ax1.twinx()
bars2 = ax2.bar(x + width/2, top10_area['hotspot_count'], width, color='darkorange', label='Focos de calor')
ax2.bar_label(bars2, labels=format_thousands(top10_area['hotspot_count']), fontsize=9, rotation=45, padding=3)
ax2.set_ylabel('Focos de calor', color='darkorange')

plt.title('TOP 10 UCs (POR ÁREA QUEIMADA AAF REAL) — ÁREA x FOCOS DE CALOR', fontweight='bold')
fig.tight_layout()
plt.show()

del aaf_area_by_uc, hotspot_count_by_uc, top10_area
gc.collect()

In [ ]:
# Top 10 UCs — quantidade de eventos por bioma predominante

uc_info = gdf_ucs_metric[['cnuc', 'nomeuc', 'bioma_pred']].drop_duplicates()

top10_events_bioma = (
    aaf_events_unique.groupby('cnuc').size().rename('n_events')
    .reset_index()
    .merge(uc_info, on='cnuc', how='left')
    .nlargest(10, 'n_events')
)

biomas_unicos = top10_events_bioma['bioma_pred'].unique()
cores_bioma = dict(zip(biomas_unicos, plt.cm.tab10.colors[:len(biomas_unicos)]))
cores_barras = top10_events_bioma['bioma_pred'].map(cores_bioma)

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#FFFDF5')
ax.set_facecolor('#FFFDF5')

bars = ax.barh(top10_events_bioma['cnuc'], top10_events_bioma['n_events'], color=cores_barras)
ax.bar_label(bars, fmt='%.0f', fontsize=11, padding=3)
ax.invert_yaxis()
ax.set_xlabel('Eventos AAF')
ax.set_title('TOP 10 UCs — QUANTIDADE DE EVENTOS AAF, POR BIOMA', fontweight='bold')
ax.grid(axis='x', linestyle='--', alpha=0.4)

legend_handles = [plt.Rectangle((0, 0), 1, 1, color=cor) for cor in cores_bioma.values()]
ax.legend(legend_handles, cores_bioma.keys(), title='Bioma', loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 UCs por eventos, comparadas ao tamanho (área total) da UC

top10_events_size = (
    aaf_events_unique.groupby('cnuc').size().rename('n_events')
    .reset_index()
    .merge(gdf_ucs_metric, on='cnuc', how='left')
    .nlargest(10, 'n_events')
)

fig, ax1 = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#FFFDF5')
ax1.set_facecolor('#FFFDF5')

x = np.arange(len(top10_events_size))
width = 0.4

bars1 = ax1.bar(x - width/2, top10_events_size['n_events'], width, color='firebrick', label='Eventos AAF')
ax1.bar_label(bars1, labels=format_thousands(top10_events_size['n_events']), fontsize=11, padding=2)
ax1.set_ylabel('Eventos AAF', color='firebrick')
ax1.set_xticks(x)
ax1.set_xticklabels(top10_events_size['cnuc'], rotation=45, ha='right')
ax1.grid(axis='y', linestyle='--', alpha=0.4)

ax2 = ax1.twinx()
bars2 = ax2.bar(x + width/2, top10_events_size['areahaalb'], width, color='steelblue', label='Área da UC (ha)')
ax2.bar_label(bars2, labels=format_thousands(top10_events_size['n_events']), fontsize=11, padding=2)
ax2.set_ylabel('Área da UC (ha)', color='steelblue')

plt.title('TOP 10 UCs COM MAIS EVENTOS AAF x TAMANHO DA UC', fontweight='bold')
fig.tight_layout()
plt.show()

### Temporal — anual, mensal, acumulado

In [ ]:
# Área queimada total, por ano

annual_area = gdf_aaf_metric.groupby('event_year')['area_ha_cell'].sum().reindex(range(2010, 2027), fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#FFFDF5')
ax.set_facecolor('#FFFDF5')

bars = ax.bar(annual_area.index.astype(str), annual_area.values, color='#8b1a1a')
ax.bar_label(bars, labels=format_thousands(annual_area.values), fontsize=11, rotation=45, padding=2)

ax.set_xlabel('Ano')
ax.set_ylabel('Área (ha)')
ax.set_title('ÁREA QUEIMADA TOTAL — ANUAL\n(Valores em hectare)', fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Área queimada total, por mês (acumulado, todos os anos)

monthly_area = gdf_aaf_metric.groupby('event_month')['area_ha_cell'].sum().reindex(range(1, 13), fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#FFFDF5')
ax.set_facecolor('#FFFDF5')

bars = ax.bar(months_label, monthly_area.values, color='#c8a951')
ax.bar_label(bars, labels=format_thousands(monthly_area.values), fontsize=11, padding=2)

ax.set_xlabel('Mês')
ax.set_ylabel('Área (ha)')
ax.set_title('ÁREA QUEIMADA TOTAL — MENSAL (ACUMULADO)\n(Valores em hectare)', fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Série mensal contínua, sem acumular anos (cada ponto = um mês de um ano específico)

min_year = aaf_events_unique['event_year'].min()
max_year = aaf_events_unique['event_year'].max()

aaf_events_unique['year_month'] = pd.to_datetime(
    aaf_events_unique['event_year'].astype(int).astype(str) + '-' +
    aaf_events_unique['event_month'].astype(int).astype(str)
).dt.to_period('M')

events_by_yearmonth = aaf_events_unique.groupby('year_month').size()

full_range = pd.period_range(start=f'{min_year}-01', end=f'{max_year}-12', freq='M')
events_by_yearmonth = events_by_yearmonth.reindex(full_range, fill_value=0)

x_labels = full_range.astype(str)

fig, ax = plt.subplots(figsize=(18, 5))
fig.patch.set_facecolor('#FFFDF5')
ax.set_facecolor('#FFFDF5')

ax.bar(x_labels, events_by_yearmonth.values, color='firebrick', alpha=0.7)
ax.set_xticks(range(0, len(x_labels), 6))
ax.set_xticklabels(x_labels[::6], rotation=45, ha='right')
ax.set_xlabel('Ano-mês')
ax.set_ylabel('Eventos AAF')
ax.set_title(f'EVENTOS AAF — SÉRIE MENSAL CONTÍNUA ({min_year}–{max_year})', fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

### Comparativo AAF x INPE (bioma, temporal)


In [ ]:
# Eventos AAF x focos INPE, por bioma

uc_bioma = gdf_ucs_metric[['cnuc', 'bioma_pred']].drop_duplicates()

aaf_by_bioma = (
    aaf_events_unique.merge(uc_bioma, on='cnuc', how='left')
    .groupby('bioma_pred').size().rename('aaf_events')
)
inpe_by_bioma = gdf_inpe_with_cell.groupby('bioma', observed=True).size().rename('inpe_hotspots')

biome_comparison = pd.concat([aaf_by_bioma, inpe_by_bioma], axis=1).fillna(0)
biome_comparison = biome_comparison.sort_values('aaf_events', ascending=False)

fig, ax1 = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor("#FFFDF5")
ax1.set_facecolor("#FFFDF5")

x = np.arange(len(biome_comparison))
width = 0.4

bars1 = ax1.bar(x - width/2, biome_comparison['aaf_events'], width, color='firebrick', label='Eventos AAF')
ax1.bar_label(bars1, labels=format_thousands(biome_comparison['aaf_events']), fontsize=11, padding=2)
ax1.set_ylabel('Eventos AAF', color='firebrick')
ax1.set_xticks(x)
ax1.set_xticklabels(biome_comparison.index, rotation=30, ha='right')
ax1.grid(axis='y', linestyle='--', alpha=0.4)

ax2 = ax1.twinx()
bars2 = ax2.bar(x + width/2, biome_comparison['inpe_hotspots'], width, color='darkorange', label='Focos INPE')
ax2.bar_label(bars2, labels=format_thousands(biome_comparison['inpe_hotspots']), fontsize=11, padding=2)
ax2.set_ylabel('Focos de calor (INPE)', color='darkorange')

plt.title('EVENTOS AAF x FOCOS DE CALOR, POR BIOMA', fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# Eventos AAF x focos INPE, por ano (mesmo intervalo do AAF)

min_year = aaf_events_unique['event_year'].min()
max_year = aaf_events_unique['event_year'].max()

aaf_by_year = aaf_events_unique.groupby('event_year').size()
inpe_by_year = gdf_inpe_with_cell[
    (gdf_inpe_with_cell['file_year'] >= min_year) & (gdf_inpe_with_cell['file_year'] <= max_year)
].groupby('file_year').size()

fig, ax1 = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor("#FFFDF5")
ax1.set_facecolor("#FFFDF5")

ax1.bar(aaf_by_year.index.astype(str), aaf_by_year.values, color='firebrick', label='Eventos AAF')
ax1.set_ylabel('Eventos AAF', color='firebrick')
ax1.grid(axis='y', linestyle='--', alpha=0.4)

ax2 = ax1.twinx()
ax2.plot(aaf_by_year.index.astype(str), inpe_by_year.reindex(aaf_by_year.index).values,
          color='darkorange', marker='o', linewidth=2, label='Focos INPE')
ax2.set_ylabel('Focos de calor (INPE)', color='darkorange')

plt.title(f'EVENTOS AAF x FOCOS INPE, POR ANO ({min_year}–{max_year})', fontweight='bold')
plt.xticks(rotation=45)
fig.tight_layout()
plt.show()


In [ ]:
# Eventos AAF x focos INPE, por mês (acumulado, mesmo intervalo do AAF)

aaf_by_month = aaf_events_unique.groupby('event_month').size().reindex(range(1, 13), fill_value=0)

gdf_inpe_with_cell['month'] = pd.to_datetime(gdf_inpe_with_cell['data_pas'], errors='coerce').dt.month
inpe_by_month = gdf_inpe_with_cell[
    (gdf_inpe_with_cell['file_year'] >= min_year) & (gdf_inpe_with_cell['file_year'] <= max_year)
].groupby('month').size().reindex(range(1, 13), fill_value=0)

fig, ax1 = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor("#FFFDF5")
ax1.set_facecolor("#FFFDF5")

ax1.bar(months_label, aaf_by_month.values, color='firebrick', label='Eventos AAF')
ax1.set_ylabel('Eventos AAF', color='firebrick')
ax1.grid(axis='y', linestyle='--', alpha=0.4)

ax2 = ax1.twinx()
ax2.plot(months_label, inpe_by_month.values, color='darkorange', marker='o', linewidth=2, label='Focos INPE')
ax2.set_ylabel('Focos de calor (INPE)', color='darkorange')

plt.title(f'EVENTOS AAF x FOCOS INPE, POR MÊS ({min_year}–{max_year}, ACUMULADO)', fontweight='bold')
fig.tight_layout()
plt.show()

### Tipo de evento harmonizado (contagem, área, FRP, espacial)


In [ ]:
# Contagem de eventos por tipo

event_type_counts = gdf_aaf_metric.dropna(subset=['event_type'])['event_type'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor("#FFFDF5")
ax.set_facecolor("#FFFDF5")

bars = ax.barh(event_type_counts.index, event_type_counts.values, color='firebrick')
ax.bar_label(bars, labels=format_thousands(event_type_counts.values), fontsize=8, padding=3)
ax.invert_yaxis()
ax.set_xlabel('Quantidade de eventos')
ax.set_title('CONTAGEM DE EVENTOS POR TIPO HARMONIZADO', fontweight='bold')
ax.grid(axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# FRP médio associado a cada tipo harmonizado (vínculo aproximado: mesma célula + ano + mês)

frp_by_cell_period = (
    gdf_inpe_with_cell.groupby(['cell_id', 'file_year', 'month'])['frp']
    .mean()
    .rename('frp_mean_period')
    .reset_index()
)

aaf_with_frp = gdf_aaf_metric.dropna(subset=['event_type']).merge(
    frp_by_cell_period,
    left_on=['cell_id', 'file_year', 'event_month'],
    right_on=['cell_id', 'file_year', 'month'],
    how='inner'
)

aaf_with_frp['event_type'].value_counts()

In [ ]:
# Total de ocorrências por tipo x FRP médio, por tipo harmonizado

occurrence_by_type = aaf_with_frp['event_type'].value_counts()
frp_mean_by_type = aaf_with_frp.groupby('event_type')['frp_mean_period'].mean()

combo_df = pd.concat([occurrence_by_type.rename('n_events'), frp_mean_by_type.rename('frp_mean')], axis=1)
combo_df = combo_df.sort_values('n_events', ascending=False)

fig, ax1 = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor("#FFFDF5")
ax1.set_facecolor("#FFFDF5")

bars = ax1.bar(combo_df.index, combo_df['n_events'], color='firebrick', alpha=0.7, label='Ocorrências')
ax1.bar_label(bars, labels=format_thousands(combo_df['n_events']), fontsize=8, padding=2)
ax1.set_ylabel('Total de ocorrências', color='firebrick')
ax1.grid(axis='y', linestyle='--', alpha=0.4)

ax2 = ax1.twinx()
ax2.plot(combo_df.index, combo_df['frp_mean'], color='darkorange', marker='o', linewidth=2, label='FRP médio')
ax2.set_ylabel('FRP médio', color='darkorange')

plt.title('TOTAL DE OCORRÊNCIAS x FRP MÉDIO, POR TIPO HARMONIZADO', fontweight='bold')
plt.xticks(rotation=30, ha='right')
fig.tight_layout()
plt.show()


### Regime do Fogo

In [9]:
#  FRP médio e máximo, por célula — todas as células com foco de calor dentro das UCs

frp_by_cell_full = gdf_inpe_with_cell.groupby('cell_id')['frp'].agg(
    frp_mean='mean', frp_max='max', n_hotspots='count'
).dropna(subset=['frp_mean'])

total_cells = len(gdf_grid)
cells_with_hotspot = gdf_inpe_with_cell['cell_id'].nunique()
cells_with_frp = len(frp_by_cell_full)
cells_no_hotspot = total_cells - cells_with_hotspot

print(f"Total de células na grade: {total_cells}")
print(f"Células com pelo menos 1 foco de calor: {cells_with_hotspot}")
print(f"Células com FRP calculável (foco válido, não nulo): {cells_with_frp}")
print(f"Células sem nenhum foco de calor registrado: {cells_no_hotspot}")

Total de células na grade: 82045
Células com pelo menos 1 foco de calor: 22550
Células com FRP calculável (foco válido, não nulo): 16489
Células sem nenhum foco de calor registrado: 59495


In [ ]:
# Histogramas — FRP médio e máximo, com estatísticas-chave marcadas

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.patch.set_facecolor("#FFFDF5")

for ax, col, color, title in [
    (axes[0], 'frp_mean', 'darkorange', 'FRP MÉDIO, POR CÉLULA'),
    (axes[1], 'frp_max', 'crimson', 'FRP MÁXIMO, POR CÉLULA'),
]:
    ax.set_facecolor("#FFFDF5")
    data = frp_by_cell_full[col]
    ax.hist(data, bins=40, color=color, alpha=0.85)

    mean_val = data.mean()
    median_val = data.median()
    ax.axvline(mean_val, color='black', linestyle='--', linewidth=1, label=f'Média: {mean_val:,.1f}'.replace(',', '.'))
    ax.axvline(median_val, color='gray', linestyle=':', linewidth=1, label=f'Mediana: {median_val:,.1f}'.replace(',', '.'))

    ax.set_xlabel(col.replace('_', ' ').upper())
    ax.set_ylabel('Quantidade de células')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

fig.suptitle(f'INTENSIDADE (FRP) — UNIVERSO COMPLETO ({cells_with_frp} células com detecção)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Comparativo — % de células por faixa de FRP médio (visão de distribuição em degraus)

frp_bins = [0, 5, 20, 50, 100, 200, 500, float('inf')]
frp_labels = ['0-5', '5-20', '20-50', '50-100', '100-200', '200-500', '500+']

frp_by_cell_full['frp_range'] = pd.cut(frp_by_cell_full['frp_mean'], bins=frp_bins, labels=frp_labels)
frp_range_counts = frp_by_cell_full['frp_range'].value_counts().reindex(frp_labels)

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor("#FFFDF5")
ax.set_facecolor("#FFFDF5")

bars = ax.bar(frp_range_counts.index.astype(str), frp_range_counts.values, color='darkorange')
ax.bar_label(bars, labels=format_thousands(frp_range_counts.values), fontsize=9, padding=3)
ax.set_xlabel('Faixa de FRP médio')
ax.set_ylabel('Quantidade de células')
ax.set_title('DISTRIBUIÇÃO DE CÉLULAS POR FAIXA DE FRP MÉDIO', fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# %% [markdown]
# ## Frequência (FRI) — apenas células com evento AAF confirmado

# %%
# Anos distintos com evento e FRI médio, por célula

years_by_cell = gdf_aaf_metric.groupby('cell_id')['event_year'].apply(lambda x: sorted(x.unique()))

def mean_return_interval(years_list):
    if len(years_list) < 2:
        return np.nan
    return np.diff(years_list).mean()

fri_by_cell = pd.DataFrame({
    'cell_id': years_by_cell.index,
    'n_fire_years': years_by_cell.apply(len),
    'fri_mean': years_by_cell.apply(mean_return_interval),
}).reset_index(drop=True)

total_cells = len(gdf_grid)
cells_with_aaf = len(fri_by_cell)
cells_never_burned = total_cells - cells_with_aaf
cells_single_year = (fri_by_cell['n_fire_years'] == 1).sum()
cells_fri_calculable = fri_by_cell['fri_mean'].notna().sum()

print(f"Total de células na grade: {total_cells}")
print(f"Células sem nenhum evento AAF: {cells_never_burned}")
print(f"Células com evento AAF em apenas 1 ano (FRI não calculável): {cells_single_year}")
print(f"Células com FRI calculável (2+ anos distintos): {cells_fri_calculable}")
print()
print("FRI médio — estatísticas (apenas células calculáveis):")
print(fri_by_cell['fri_mean'].describe())

# %%
# Histograma do FRI médio, com estatísticas-chave marcadas

fig, ax = plt.subplots(figsize=(11, 5.5))
fig.patch.set_facecolor("#FFFDF5")
ax.set_facecolor("#FFFDF5")

fri_valid = fri_by_cell['fri_mean'].dropna()
ax.hist(fri_valid, bins=30, color='firebrick', alpha=0.85)

mean_val = fri_valid.mean()
median_val = fri_valid.median()
ax.axvline(mean_val, color='black', linestyle='--', linewidth=1, label=f'Média: {mean_val:.1f} anos')
ax.axvline(median_val, color='gray', linestyle=':', linewidth=1, label=f'Mediana: {median_val:.1f} anos')

ax.set_xlabel('FRI médio (anos)')
ax.set_ylabel('Quantidade de células')
ax.set_title(f'FREQUÊNCIA (FRI) — INTERVALO MÉDIO DE RETORNO ({cells_fri_calculable} células com AAF confirmado)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# %%
# Distribuição de quantas vezes (anos distintos) cada célula queimou — robustez do FRI

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor("#FFFDF5")
ax.set_facecolor("#FFFDF5")

ax.hist(fri_by_cell['n_fire_years'], bins=range(1, 18), color='teal', edgecolor='white')
ax.set_xlabel('Quantidade de anos distintos com evento')
ax.set_ylabel('Quantidade de células')
ax.set_title('ROBUSTEZ DO FRI — QUANTOS ANOS DISTINTOS SUSTENTAM CADA CÁLCULO', fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# %%
del years_by_cell, frp_by_cell_full, frp_range_counts, fri_by_cell, fri_valid
gc.collect()

In [ ]:
#  Sazonalidade — janela de meses concentrando 80% dos eventos, por UC

def seasonal_window(months, coverage=0.8):
    # ordena meses pela frequência, acumula até atingir a cobertura desejada, retorna quantos meses foram necessários
    counts = months.value_counts().sort_values(ascending=False)
    cumulative = counts.cumsum() / counts.sum()
    return (cumulative <= coverage).sum() + 1

seasonal_by_uc = (
    gdf_aaf_metric.dropna(subset=['cnuc'])
    .groupby('cnuc')['event_month']
    .apply(lambda x: seasonal_window(x, 0.8))
    .rename('n_months_80pct')
)

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#FFFDF5')
ax.set_facecolor('#FFFDF5')

ax.hist(seasonal_by_uc, bins=range(1, 13), color='steelblue', edgecolor='white')
ax.set_xlabel('Meses necessários para cobrir 80% dos eventos')
ax.set_ylabel('Quantidade de UCs')
ax.set_title('SAZONALIDADE — JANELA DE MESES CONCENTRANDO 80% DOS EVENTOS, POR UC', fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Tamanho — máximo por célula, e distribuição por tipo de evento harmonizado

# %%
max_size_by_cell = gdf_aaf_metric.groupby('cell_id')['area_ha_cell'].max()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#FFFDF5')

axes[0].set_facecolor('#FFFDF5')
axes[0].hist(max_size_by_cell, bins=30, color='firebrick')
axes[0].set_xlabel('Tamanho máximo do evento (ha)')
axes[0].set_ylabel('Quantidade de células')
axes[0].set_title('TAMANHO MÁXIMO DE EVENTO, POR CÉLULA')
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

axes[1].set_facecolor('#FFFDF5')
data_by_type = [
    gdf_aaf_metric[gdf_aaf_metric['event_type'] == t]['area_ha_cell'].dropna()
    for t in gdf_aaf_metric['event_type'].dropna().unique()
]
axes[1].boxplot(data_by_type, labels=gdf_aaf_metric['event_type'].dropna().unique(), showfliers=False)
axes[1].set_ylabel('Área do evento (ha)')
axes[1].set_title('DISTRIBUIÇÃO DE TAMANHO, POR TIPO HARMONIZADO')
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Área média anual queimada — por célula, comparada ao tamanho da célula (2.500 ha)

# %%
CELL_AREA_HA = 2500  # 5km x 5km

annual_area_by_cell = gdf_aaf_metric.groupby(['cell_id', 'event_year'])['area_ha_cell'].sum().reset_index()
mean_annual_area_by_cell = annual_area_by_cell.groupby('cell_id')['area_ha_cell'].mean()
pct_of_cell_burned = (mean_annual_area_by_cell / CELL_AREA_HA * 100).clip(upper=100)

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#FFFDF5')
ax.set_facecolor('#FFFDF5')

ax.hist(pct_of_cell_burned, bins=30, color='saddlebrown')
ax.axvline(50, color='darkred', linestyle='--', label='50% da célula')
ax.set_xlabel('% da área da célula queimada, em média, por ano')
ax.set_ylabel('Quantidade de células')
ax.set_title('ÁREA MÉDIA ANUAL QUEIMADA — % DO TAMANHO DA CÉLULA (2.500 ha)', fontweight='bold')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print("Células que queimam, em média, mais de 50% de sua área todo ano:", (pct_of_cell_burned > 50).sum())


In [ ]:
# Limpeza de memória

del (
    aaf_area_by_uc,
    hotspot_count_by_uc,
    top10_area,
    uc_info,
    top10_events_bioma,
    biomas_unicos,
    cores_bioma,
    cores_barras,
    top10_events_size,
    annual_area,
    monthly_area,
    events_by_yearmonth,
    full_range,
    x_labels,
    uc_bioma,
    aaf_by_bioma,
    inpe_by_bioma,
    biome_comparison,
    aaf_by_year,
    inpe_by_year,
    aaf_by_month,
    inpe_by_month,
    event_type_counts,
    frp_by_cell_period,
    aaf_with_frp,
    frp_by_cell_period,
    aaf_with_frp,
    occurrence_by_type,
    frp_mean_by_type,
    combo_df,
    seasonal_by_uc,
    max_size_by_cell,
    data_by_type,
    annual_area_by_cell,
    mean_annual_area_by_cell,
    pct_of_cell_burned,
)
gc.collect()

# Pré-processamento

In [37]:
# Universo final: todas as células da grade
TOTAL_CELLS = len(gdf_grid)
print("Total de células:", TOTAL_CELLS)

Total de células: 82045


In [40]:
# Thresholds de robustez: mínimo de observações para considerar cada variável confiável

MIN_YEARS_FOR_FRI = 3  # anos distintos com evento
MIN_HOTSPOTS_FOR_FRP = 5  # focos de calor detectados
MIN_EVENTS_FOR_SEASONALITY = 3
MIN_YEARS_FOR_AREA = 3

In [ ]:
# FRI — intervalo médio de retorno (AAF)


# %%
def mean_return_interval(years_list):
    # média dos intervalos entre anos consecutivos com evento; exige 2+ anos
    if len(years_list) < 2:
        return np.nan
    return np.diff(years_list).mean()


years_by_cell = gdf_aaf_metric.groupby("cell_id")["event_year"].apply(
    lambda x: sorted(x.unique())
)

fri_raw = pd.DataFrame(
    {
        "cell_id": years_by_cell.index,
        "n_fire_years": years_by_cell.apply(len),
        "fri_mean": years_by_cell.apply(mean_return_interval),
    }
).reset_index(drop=True)

fri_raw["fri_reliable"] = fri_raw["n_fire_years"] >= MIN_YEARS_FOR_FRI

print("Células com evento AAF:", len(fri_raw))
print(
    f"Células com FRI confiável (>={MIN_YEARS_FOR_FRI} anos):",
    fri_raw["fri_reliable"].sum(),
)

Células com evento AAF: 6666
Células com FRI confiável (>=3 anos): 3191


In [ ]:
# FRP — intensidade média e máxima (INPE)

# %%
frp_raw = (
    gdf_inpe_with_cell.groupby("cell_id")["frp"]
    .agg(frp_mean="mean", frp_max="max", n_hotspots="count")
    .reset_index()
    .dropna(subset=["frp_mean"])
)

frp_raw["frp_reliable"] = frp_raw["n_hotspots"] >= MIN_HOTSPOTS_FOR_FRP

print("Células com FRP calculável:", len(frp_raw))
print(
    f"Células com FRP confiável (>={MIN_HOTSPOTS_FOR_FRP} hotspots):",
    frp_raw["frp_reliable"].sum(),
)

Células com FRP calculável: 16489
Células com FRP confiável (>=5 hotspots): 11889


In [42]:
#  Sazonalidade — janela de meses concentrando 80% dos eventos (AAF)
def seasonal_window(months, coverage=0.8):
    # quantos meses (do mais ativo ao menos ativo) somam a cobertura desejada
    counts = months.value_counts().sort_values(ascending=False)
    cumulative = counts.cumsum() / counts.sum()
    return (cumulative <= coverage).sum() + 1

seasonality_raw = (
    gdf_aaf_metric.groupby('cell_id')['event_month']
    .agg(n_months_80pct=lambda x: seasonal_window(x, 0.8), n_events='size')
    .reset_index()
)

seasonality_raw['seasonality_reliable'] = seasonality_raw['n_events'] >= MIN_EVENTS_FOR_SEASONALITY

print("Células com sazonalidade calculável:", len(seasonality_raw))
print(f"Células com sazonalidade confiável (>={MIN_EVENTS_FOR_SEASONALITY} eventos):", seasonality_raw['seasonality_reliable'].sum())



Células com sazonalidade calculável: 6666
Células com sazonalidade confiável (>=3 eventos): 4256


In [ ]:
# Tamanho — maior evento por célula (AAF)

size_raw = (
    gdf_aaf_metric.groupby('cell_id')['area_ha_cell']
    .agg(size_max='max', n_events='size')
    .reset_index()
)

print("Células com evento AAF:", len(size_raw))

Células com evento AAF: 6666


In [45]:
# Área média anual queimada (AAF)

annual_area_by_year = gdf_aaf_metric.groupby(['cell_id', 'event_year'])['area_ha_cell'].sum().reset_index()

annual_area_raw = (
    annual_area_by_year.groupby('cell_id')['area_ha_cell']
    .agg(annual_area_mean='mean', n_years='size')
    .reset_index()
)

annual_area_raw['area_reliable'] = annual_area_raw['n_years'] >= MIN_YEARS_FOR_AREA

print("Células com área anual calculável:", len(annual_area_raw))
print(f"Células com área anual confiável (>={MIN_YEARS_FOR_AREA} anos):", annual_area_raw['area_reliable'].sum())

Células com área anual calculável: 6666
Células com área anual confiável (>=3 anos): 3191


In [ ]:
# União das 5 variáveis num único vetor por célula, cobrindo o universo completo (82.045)

fire_regime_vector = gdf_grid[['cell_id']].copy()

fire_regime_vector = fire_regime_vector.merge(
    fri_raw[['cell_id', 'fri_mean', 'n_fire_years', 'fri_reliable']], on='cell_id', how='left'
)
fire_regime_vector = fire_regime_vector.merge(
    frp_raw[['cell_id', 'frp_mean', 'frp_max', 'n_hotspots', 'frp_reliable']], on='cell_id', how='left'
)
fire_regime_vector = fire_regime_vector.merge(
    seasonality_raw[['cell_id', 'n_months_80pct', 'seasonality_reliable']], on='cell_id', how='left'
)
fire_regime_vector = fire_regime_vector.merge(
    size_raw[['cell_id', 'size_max']], on='cell_id', how='left'
)
fire_regime_vector = fire_regime_vector.merge(
    annual_area_raw[['cell_id', 'annual_area_mean', 'area_reliable']], on='cell_id', how='left'
)

print("Total de células no vetor final:", len(fire_regime_vector))
fire_regime_vector.head(10)

Total de células no vetor final: 82045


,cell_id,fri_mean,n_fire_years,fri_reliable,frp_mean,frp_max,n_hotspots,frp_reliable,n_months_80pct,seasonality_reliable,size_max,annual_area_mean,area_reliable
0,0_575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0_576,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0_579,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0_580,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1000_189,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1000_190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1000_191,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1000_192,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1000_193,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1000_194,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Codificar ausência de dado por variável, conforme discutido:
# - FRP, tamanho, área média: zero tem significado físico real (nunca houve fogo/energia detectada)
# - FRI e sazonalidade: não existe número honesto para "nunca aconteceu" — viram categoria à parte

fire_regime_vector['frp_mean'] = fire_regime_vector['frp_mean'].fillna(0)
fire_regime_vector['frp_max'] = fire_regime_vector['frp_max'].fillna(0)
fire_regime_vector['size_max'] = fire_regime_vector['size_max'].fillna(0)
fire_regime_vector['annual_area_mean'] = fire_regime_vector['annual_area_mean'].fillna(0)

# Categoria de status para FRI — separa "sem registro nenhum" de "registro insuficiente" de "confiável"
def fri_status(row):
    if pd.isna(row['n_fire_years']):
        return 'sem_registro'
    elif not row['fri_reliable']:
        return 'insuficiente'
    else:
        return 'confiavel'

fire_regime_vector['fri_status'] = fire_regime_vector.apply(fri_status, axis=1)

# Mesma lógica para sazonalidade
def seasonality_status(row):
    if pd.isna(row['n_months_80pct']):
        return 'sem_registro'
    elif not row['seasonality_reliable']:
        return 'insuficiente'
    else:
        return 'confiavel'

fire_regime_vector['seasonality_status'] = fire_regime_vector.apply(seasonality_status, axis=1)

# Preencher os flags booleanos restantes (frp_reliable) onde ficaram NaN
fire_regime_vector['frp_reliable'] = fire_regime_vector['frp_reliable'].fillna(False)
fire_regime_vector['area_reliable'] = fire_regime_vector['area_reliable'].fillna(False)

print(fire_regime_vector[['fri_status', 'seasonality_status']].apply(lambda col: col.value_counts()))
fire_regime_vector.head(10)

              fri_status  seasonality_status
confiavel           3191                4256
insuficiente        3475                2410
sem_registro       75379               75379


,cell_id,fri_mean,n_fire_years,fri_reliable,frp_mean,frp_max,n_hotspots,frp_reliable,n_months_80pct,seasonality_reliable,size_max,annual_area_mean,area_reliable,fri_status,seasonality_status
0,0_575,NaN,NaN,NaN,0.0,0.0,NaN,False,NaN,NaN,0.0,0.0,False,sem_registro,sem_registro
1,0_576,NaN,NaN,NaN,0.0,0.0,NaN,False,NaN,NaN,0.0,0.0,False,sem_registro,sem_registro
2,0_579,NaN,NaN,NaN,0.0,0.0,NaN,False,NaN,NaN,0.0,0.0,False,sem_registro,sem_registro
3,0_580,NaN,NaN,NaN,0.0,0.0,NaN,False,NaN,NaN,0.0,0.0,False,sem_registro,sem_registro
4,1000_189,NaN,NaN,NaN,0.0,0.0,NaN,False,NaN,NaN,0.0,0.0,False,sem_registro,sem_registro
5,1000_190,NaN,NaN,NaN,0.0,0.0,NaN,False,NaN,NaN,0.0,0.0,False,sem_registro,sem_registro
6,1000_191,NaN,NaN,NaN,0.0,0.0,NaN,False,NaN,NaN,0.0,0.0,False,sem_registro,sem_registro
7,1000_192,NaN,NaN,NaN,0.0,0.0,NaN,False,NaN,NaN,0.0,0.0,False,sem_registro,sem_registro
8,1000_193,NaN,NaN,NaN,0.0,0.0,NaN,False,NaN,NaN,0.0,0.0,False,sem_registro,sem_registro
9,1000_194,NaN,NaN,NaN,0.0,0.0,NaN,False,NaN,NaN,0.0,0.0,False,sem_registro,sem_registro


In [ ]:
# Segmentação em dois grupos, conforme decidido:
# Grupo A — confiável nas 5 variáveis simultaneamente, entra no clustering real
# Grupo B — o resto, recebe classificação fixa de "regime raro/sem dado suficiente"

fire_regime_vector['group'] = np.where(
    (fire_regime_vector['fri_status'] == 'confiavel') &
    (fire_regime_vector['frp_reliable'] == True) &
    (fire_regime_vector['seasonality_status'] == 'confiavel') &
    (fire_regime_vector['area_reliable'] == True),
    'A_confiavel',
    'B_insuficiente'
)

print(fire_regime_vector['group'].value_counts())
print(f"\nPercentual no Grupo A: {(fire_regime_vector['group'] == 'A_confiavel').mean() * 100:.1f}%")

group
B_insuficiente    78907
A_confiavel        3138
Name: count, dtype: int64

Percentual no Grupo A: 3.8%


In [59]:
fire_regime_vector['group'].describe()


count              82045
unique                 2
top       B_insuficiente
freq               78907
Name: group, dtype: object

In [ ]:
# %%
# Isolar o Grupo A para normalização e clustering

group_a = fire_regime_vector[fire_regime_vector['group'] == 'A_confiavel'].copy()

print("Células no Grupo A:", len(group_a))
print(group_a[['fri_mean', 'frp_mean', 'n_months_80pct', 'size_max', 'annual_area_mean']].describe())


Células no Grupo A: 3138
          fri_mean     frp_mean  n_months_80pct     size_max  annual_area_mean
count  3138.000000  3138.000000     3138.000000  3138.000000       3138.000000
mean      2.147649    38.187996        3.094009   973.851391        601.349235
std       1.268486    24.966661        1.115467   730.767809        474.112202
min       1.000000     2.240741        1.000000     5.953380          3.384906
25%       1.200000    21.609262        2.000000   348.465590        215.980272
50%       1.666667    33.362038        3.000000   793.807726        493.424978
75%       2.666667    48.750138        4.000000  1523.527112        871.075226
max       7.500000   336.644989        7.000000  2500.000000       2632.357710


In [ ]:
# Normalização — log1p nas variáveis com cauda longa, depois z-score em todas

from scipy import stats

# Colunas com forte assimetria (desvio padrão > média, cauda longa) recebem log1p antes do z-score
log_transform_cols = ['frp_mean', 'size_max', 'annual_area_mean']
direct_zscore_cols = ['fri_mean', 'n_months_80pct']

group_a_norm = group_a[['cell_id']].copy()

for col in log_transform_cols:
    logged = np.log1p(group_a[col])
    group_a_norm[f'{col}_norm'] = (logged - logged.mean()) / logged.std()

for col in direct_zscore_cols:
    group_a_norm[f'{col}_norm'] = (group_a[col] - group_a[col].mean()) / group_a[col].std()

print(group_a_norm.describe())
group_a_norm.head(10)

       frp_mean_norm  size_max_norm  annual_area_mean_norm  fri_mean_norm  n_months_80pct_norm
count   3.138000e+03   3.138000e+03           3.138000e+03   3.138000e+03         3.138000e+03
mean   -7.293876e-09   1.811454e-17          -6.611806e-16  -4.528634e-17        -5.434361e-17
std     1.000000e+00   1.000000e+00           1.000000e+00   1.000000e+00         1.000000e+00
min    -3.947199e+00  -4.211102e+00          -4.224885e+00  -9.047397e-01        -1.877248e+00
25%    -6.481067e-01  -5.671224e-01          -5.654783e-01  -7.470714e-01        -9.807628e-01
50%     6.279901e-02   1.972649e-01           2.069784e-01  -3.791787e-01        -8.427761e-02
75%     6.912822e-01   8.031775e-01           7.392268e-01   4.091629e-01         8.122076e-01
max     3.943541e+00   1.263661e+00           1.775754e+00   4.219480e+00         3.501663e+00


,cell_id,frp_mean_norm,size_max_norm,annual_area_mean_norm,fri_mean_norm,n_months_80pct_norm
12289,108_511,-0.330820,-3.896303,-3.789097,1.460285,-0.084278
12329,109_511,-0.271972,-2.060630,-1.663218,1.460285,-0.084278
12529,114_508,-0.630813,-4.211102,-4.224885,1.460285,-0.980763
12530,114_509,-0.551441,-3.061558,-3.102672,1.460285,-0.980763
12600,117_507,-0.503995,-0.577252,-0.721326,1.460285,-0.980763
12625,118_507,-0.863485,-1.637301,-1.603275,1.460285,-0.084278
12626,118_508,-0.874509,-4.153980,-4.141694,1.066114,-0.084278
12725,120_510,-0.066540,-3.222541,-3.324812,1.066114,-0.084278
13108,130_517,-0.833987,-3.250764,-3.417488,1.066114,-0.084278
13231,133_516,0.467123,-3.474040,-3.511901,1.066114,-0.084278


In [64]:
# %%
# K-means — testar K de 3 a 8, usando inércia (cotovelo) e silhouette score

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

feature_cols = ['frp_mean_norm', 'size_max_norm', 'annual_area_mean_norm', 'fri_mean_norm', 'n_months_80pct_norm']
X = group_a_norm[feature_cols].values

results = []

for k in range(3, 9):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)

    inertia = kmeans.inertia_
    silhouette = silhouette_score(X, labels)

    results.append({'k': k, 'inertia': inertia, 'silhouette': silhouette})
    print(f"K={k} | Inércia={inertia:.1f} | Silhouette={silhouette:.3f}")

results_df = pd.DataFrame(results)

: 

In [ ]:

# %%
del years_by_cell, annual_area_by_year
gc.collect()

# Modelagem / treinamento

# Avaliação do modelo


# Ajuste de hiperparâmetros


# Validação final
